In [ ]:
# ==== Imports & config ====
import os, math, random, argparse
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Paths (change if your structure differs)
DATA_DIR = "../../data"   # folder with train.csv, val.csv, test.csv, metaData.csv
SUBMIT_PATH = "./submission.csv"

NUM_LABELS = 388
LABEL_COLUMNS = [str(i) for i in range(NUM_LABELS)]

ROOM_CATEGORIES = [
    "andre områder","kjøkken","stue","gang","soverom",
    "bad","bod","vaskerom","wc","kjeller","garasje"
]
ROOM_INDEX = {r:i for i,r in enumerate(ROOM_CATEGORIES)}
USE_ROOM_ONEHOT = False  # flip to True to include room one-hot (~11 dims)

# ==== Hackathon metric (standalone) ====
def get_room_scores(room_preds: List[int], room_targets: List[int]) -> Dict[str, float]:
    EMPTY_ROOM_REWARD = 1
    TRUE_POSITIVE_REWARD = 1
    FALSE_POSITIVE_PENALTY = 0.25
    FALSE_NEGATIVE_PENALTY = 0.5

    best_possible_score = 0.0
    dummy_score = 0.0
    score = 0.0

    room_targets_set = set(room_targets)
    room_preds_set = set(room_preds)

    if len(room_targets_set) == 0:
        best_possible_score += EMPTY_ROOM_REWARD
        dummy_score += EMPTY_ROOM_REWARD
        score += EMPTY_ROOM_REWARD if len(room_preds_set) == 0 else -FALSE_POSITIVE_PENALTY * len(room_preds_set)
    else:
        best_possible_score += len(room_targets_set) * TRUE_POSITIVE_REWARD
        dummy_score -= len(room_targets_set) * FALSE_NEGATIVE_PENALTY
        score += TRUE_POSITIVE_REWARD * len(room_targets_set & room_preds_set)
        score -= FALSE_POSITIVE_PENALTY * len(room_preds_set - room_targets_set)
        score -= FALSE_NEGATIVE_PENALTY * len(room_targets_set - room_preds_set)

    if best_possible_score == dummy_score:
        normalized_score = -abs(score)
    else:
        normalized_score = (score - dummy_score) / (best_possible_score - dummy_score)

    return {"score": score, "dummy_score": dummy_score,
            "best_possible_score": best_possible_score, "normalized_score": normalized_score}

def normalized_rooms_score(preds: List[List[int]], targets: List[List[int]]) -> float:
    best_possible_score = 0.0
    dummy_score = 0.0
    score = 0.0
    for room_targets, room_preds in zip(targets, preds, strict=True):
        rs = get_room_scores(room_preds, room_targets)
        score += rs["score"]; dummy_score += rs["dummy_score"]; best_possible_score += rs["best_possible_score"]
    if best_possible_score == dummy_score:
        return -abs(score)
    return (score - dummy_score) / (best_possible_score - dummy_score)


In [ ]:
# ==== Load ====
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
meta  = pd.read_csv(os.path.join(DATA_DIR, "metaData.csv"))

for df in (train, val, test, meta):
    df["project_id"] = df["project_id"].astype(int)

# ==== Join ====
train_joined = train.merge(meta, on="project_id", how="left", suffixes=("_train", "_meta"))
val_joined   = val.merge(meta,   on="project_id", how="left", suffixes=("_train", "_meta"))
test_joined  = test.merge(meta,  on="project_id", how="left", suffixes=("_train", "_meta"))

# ==== Seasonality features ====
def add_month_features(df: pd.DataFrame) -> pd.DataFrame:
    if "case_creation_month" not in df.columns:
        raise KeyError("case_creation_month missing. Ensure month features are added BEFORE pruning columns.")
    m = pd.to_numeric(df["case_creation_month"], errors="coerce").fillna(6).astype(int).clip(1, 12)
    angle = 2 * np.pi * (m - 1) / 12.0
    out = df.copy()
    out["case_creation_month"] = m
    out["month_sin"] = np.sin(angle).astype(np.float32)
    out["month_cos"] = np.cos(angle).astype(np.float32)
    return out

train_joined = add_month_features(train_joined)
val_joined   = add_month_features(val_joined)
test_joined  = add_month_features(test_joined)

# Keep only essentials (incl month features)
cols_keep = [
    "id", "project_id", "room", "work_operation_cluster_code",
    "case_creation_month", "month_sin", "month_cos"
]
train_joined = train_joined[cols_keep].copy()
val_joined   = val_joined[cols_keep].copy()
test_joined  = test_joined[cols_keep].copy()

print(train_joined.head(3))


In [ ]:
# ==== Rooms & vector helpers ====
ROOM_CATEGORIES = [
    "andre områder","kjøkken","stue","gang","soverom",
    "bad","bod","vaskerom","wc","kjeller","garasje"
]
ROOM_INDEX = {r:i for i,r in enumerate(ROOM_CATEGORIES)}

def map_room(room_name: str) -> str:
    if not isinstance(room_name, str): return "ukjent"
    s = room_name.lower()
    for r in ROOM_CATEGORIES:
        if r in s: return r
    return "ukjent"

def multi_hot_from_codes(codes: List[int], num_labels: int = NUM_LABELS) -> np.ndarray:
    v = np.zeros(num_labels, dtype=np.float32)
    for c in codes:
        if 0 <= int(c) < num_labels: v[int(c)] = 1.0
    return v

def mask_codes_for_training(full_codes: List[int],
                            min_hide:int=1, max_frac:float=0.5,
                            rng: np.random.Generator | None = None) -> Tuple[List[int], List[int]]:
    rng = rng or np.random.default_rng(SEED)
    S = sorted(set(int(x) for x in full_codes))
    if len(S) <= 1: return S, []  # can't hide the only one
    max_hide = max(min_hide, int(math.floor(len(S) * max_frac)))
    max_hide = min(max_hide, len(S)-1)
    n_hide = int(rng.integers(low=min_hide, high=max_hide+1))
    H = sorted(rng.choice(S, size=n_hide, replace=False).tolist())
    O = [x for x in S if x not in H]
    return O, H

# ==== Aggregate to one row per id ====
def aggregate_per_id(df: pd.DataFrame) -> pd.DataFrame:
    g = (
        df.groupby(["project_id", "room", "id"])
          .agg({
              "work_operation_cluster_code": list,
              "case_creation_month": "first",
              "month_sin": "first",
              "month_cos": "first",
          })
          .reset_index()
          .rename(columns={"work_operation_cluster_code": "codes"})
    )
    g["room_category"] = g["room"].apply(map_room)
    return g

train_agg = aggregate_per_id(train_joined)
val_agg   = aggregate_per_id(val_joined)
test_agg  = aggregate_per_id(test_joined)

# ==== Build X/Y matrices from aggregated rows ====
USE_ROOM_ONEHOT = False  # toggle if you want room one-hot appended

def build_xy_from_agg(df_agg: pd.DataFrame, do_mask: bool, rng: np.random.Generator):
    X_list, Y_list, meta_rows = [], [], []
    for _, row in df_agg.iterrows():
        rid = int(row["id"])
        codes_full = [int(c) for c in row["codes"]]
        room_cat = row["room_category"]

        observed, hidden = (mask_codes_for_training(codes_full, rng=rng) if do_mask
                            else (codes_full, []))

        x_ops = multi_hot_from_codes(observed, NUM_LABELS)

        if USE_ROOM_ONEHOT:
            rc_vec = np.zeros(len(ROOM_CATEGORIES), dtype=np.float32)
            rc_idx = ROOM_INDEX.get(room_cat, None)
            if rc_idx is not None: rc_vec[rc_idx] = 1.0
            x_vec = np.concatenate([x_ops, rc_vec], axis=0)
        else:
            x_vec = x_ops

        # append seasonal features
        ms = np.float32(row["month_sin"]); mc = np.float32(row["month_cos"])
        x_vec = np.concatenate([x_vec, np.array([ms, mc], dtype=np.float32)], axis=0)

        y_vec = multi_hot_from_codes(hidden, NUM_LABELS)

        X_list.append(x_vec); Y_list.append(y_vec)
        meta_rows.append({
            "id": rid, "project_id": int(row["project_id"]), "room_category": room_cat,
            "observed_codes": observed, "hidden_codes": hidden,
            "month_sin": float(ms), "month_cos": float(mc),
        })

    X = np.stack(X_list, axis=0).astype(np.float32)
    Y = np.stack(Y_list, axis=0).astype(np.float32)
    meta = pd.DataFrame(meta_rows)
    return X, Y, meta

rng_train = np.random.default_rng(SEED)
rng_val   = np.random.default_rng(SEED + 1)

X_train, Y_train, meta_train = build_xy_from_agg(train_agg, do_mask=True, rng=rng_train)
X_val,   Y_val,   meta_val   = build_xy_from_agg(val_agg,   do_mask=True, rng=rng_val)

print("Train shapes:", X_train.shape, Y_train.shape)
print("Val   shapes:", X_val.shape,   Y_val.shape)


In [ ]:
# ==== Dataset & loaders ====
class RoomsDataset(Dataset):
    def __init__(self, X: np.ndarray, Y: np.ndarray):
        self.X = X.astype(np.float32)
        self.Y = Y.astype(np.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.Y[idx])

train_ds = RoomsDataset(X_train, Y_train)
val_ds   = RoomsDataset(X_val,   Y_val)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False)

# ==== Model ====
class MLP(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, output_dim)  # logits
        )
    def forward(self, x): return self.net(x)

INPUT_DIM  = X_train.shape[1]
OUTPUT_DIM = NUM_LABELS
model = MLP(INPUT_DIM, OUTPUT_DIM).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ==== Train ====
def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, total_n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)
        if train_mode:
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * xb.size(0); total_n += xb.size(0)
    return total_loss / max(1, total_n)

EPOCHS = 10
for epoch in range(1, EPOCHS+1):
    tr = run_epoch(model, train_loader, criterion, optimizer)
    va = run_epoch(model, val_loader,   criterion, optimizer=None)
    print(f"Epoch {epoch:02d} | train {tr:.4f} | val {va:.4f}")


In [ ]:
# ==== Gather validation probabilities ====
model.eval()
with torch.no_grad():
    probs_val_parts = []
    for xb, _ in val_loader:
        logits = model(xb.to(DEVICE))
        probs_val_parts.append(torch.sigmoid(logits).cpu().numpy())
    probs_val = np.concatenate(probs_val_parts, axis=0)

targets_val  = meta_val["hidden_codes"].tolist()
observed_val = meta_val["observed_codes"].tolist()

# ==== Threshold tuning (coordinate descent over labels) ====
def preds_from_probs(probs: np.ndarray, thresholds: np.ndarray, observed_lists: List[List[int]]) -> List[List[int]]:
    bin_mat = (probs >= thresholds[None, :]).astype(np.int32)
    preds = []
    for i in range(bin_mat.shape[0]):
        idx = bin_mat[i].nonzero()[0].tolist()
        obs = set(int(c) for c in observed_lists[i])
        preds.append([j for j in idx if j not in obs])
    return preds

def score_with_thresholds(probs, thresholds, observed, targets) -> float:
    return normalized_rooms_score(preds_from_probs(probs, thresholds, observed), targets)

def tune_thresholds_coordinate_descent(
    probs: np.ndarray, observed: List[List[int]], targets: List[List[int]],
    init: float = 0.5, grid = np.linspace(0.2, 0.8, 7), max_passes: int = 2
) -> np.ndarray:
    ths = np.full((probs.shape[1],), init, dtype=np.float32)
    base = score_with_thresholds(probs, ths, observed, targets)
    print(f"[tune] start score: {base:.4f}")
    for p in range(max_passes):
        improved = False
        for j in range(probs.shape[1]):
            best_t, best_s = ths[j], base
            for t in grid:
                if t == ths[j]: continue
                ths_try = ths.copy(); ths_try[j] = t
                s = score_with_thresholds(probs, ths_try, observed, targets)
                if s > best_s: best_s, best_t = s, t
            if best_t != ths[j]:
                ths[j] = best_t; base = best_s; improved = True
        print(f"[tune] pass {p+1}: score {base:.4f}")
        if not improved: break
    return ths

thresholds_vec = tune_thresholds_coordinate_descent(
    probs=probs_val, observed=observed_val, targets=targets_val,
    init=0.5, grid=np.linspace(0.2,0.8,7), max_passes=2
)
print("Validation (tuned) score:", score_with_thresholds(probs_val, thresholds_vec, observed_val, targets_val))


In [ ]:
# ==== Build test inputs (no masking) ====
X_test_ops, test_ids, observed_by_id = [], [], {}
for _, row in test_agg.iterrows():
    rid = int(row["id"])
    codes = [int(c) for c in row["codes"]]
    observed_by_id[rid] = sorted(set(codes))

    x_vec = multi_hot_from_codes(codes, NUM_LABELS)

    if USE_ROOM_ONEHOT:
        rc_vec = np.zeros(len(ROOM_CATEGORIES), dtype=np.float32)
        rc_idx = ROOM_INDEX.get(row["room_category"], None)
        if rc_idx is not None: rc_vec[rc_idx] = 1.0
        x_vec = np.concatenate([x_vec, rc_vec], axis=0)

    # append month features to test too
    ms = np.float32(row["month_sin"]); mc = np.float32(row["month_cos"])
    x_vec = np.concatenate([x_vec, np.array([ms, mc], dtype=np.float32)], axis=0)

    X_test_ops.append(x_vec); test_ids.append(rid)

X_test = np.stack(X_test_ops, axis=0).astype(np.float32)

# ==== Predict ====
model.eval()
with torch.no_grad():
    logits_test = model(torch.from_numpy(X_test).to(DEVICE))
    probs_test  = torch.sigmoid(logits_test).cpu().numpy()

# Apply tuned per-label thresholds
pred_bin = (probs_test >= thresholds_vec[None, :]).astype(np.int32)

# Never predict already-observed ops
for i, rid in enumerate(test_ids):
    for c in observed_by_id[rid]:
        if 0 <= c < NUM_LABELS:
            pred_bin[i, c] = 0

# ==== (Optional) simple room-wise top-K clip ====
USE_ROOM_TOPK = False
DEFAULT_TOPK = 5
if USE_ROOM_TOPK:
    from collections import defaultdict
    room2counts = defaultdict(list)
    for rc, tgt in zip(meta_val["room_category"], targets_val):
        room2counts[rc].append(len(tgt))
    room2k = {rc: max(1, int(np.median(cnts))) for rc, cnts in room2counts.items()}

    for i in range(pred_bin.shape[0]):
        rc = test_agg.iloc[i]["room_category"]
        k = room2k.get(rc, DEFAULT_TOPK)
        pos_idx = np.where(pred_bin[i] == 1)[0]
        if len(pos_idx) > k:
            keep = pos_idx[np.argsort(-probs_test[i, pos_idx])[:k]]
            drop_idx = pos_idx[~np.isin(pos_idx, keep)]
            pred_bin[i, drop_idx] = 0

# ==== Submission ====
submission = pd.DataFrame(pred_bin, columns=LABEL_COLUMNS)
submission.insert(0, "id", test_ids)
for c in LABEL_COLUMNS: submission[c] = submission[c].astype(int)
submission = submission.sort_values("id").reset_index(drop=True)

submission.to_csv(SUBMIT_PATH, index=False)
print(f"Saved submission to: {SUBMIT_PATH}")
submission.head()
